In [0]:
dbutils.widgets.text("catalogo", "mlops", "Catálogo")
dbutils.widgets.text("esquema", "frecuencia", "Esquema")
dbutils.widgets.text("modelo", "frecuencia_rc", "Modelo")
dbutils.widgets.text("experimento_validacion", "/Users/santiagopadu@gmail.com/frecuencia_rc_validacion", "Experimento validación")
dbutils.widgets.text("mejora_minima", "0.01", "Mejora mínima devianza")
dbutils.widgets.text("tolerancia_ae", "0.02", "Tolerancia A/E")

catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
modelo_nombre = dbutils.widgets.get("modelo")
experimento_validacion = dbutils.widgets.get("experimento_validacion")
mejora_minima = float(dbutils.widgets.get("mejora_minima"))
tolerancia_ae = float(dbutils.widgets.get("tolerancia_ae"))

In [0]:
import mlflow
import pandas as pd
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from sklearn.metrics import mean_poisson_deviance

mlflow.sklearn.autolog(disable=True)
cliente = MlflowClient()
nombre_modelo = f"{catalogo}.{esquema}.{modelo_nombre}"
tabla_test = f"{catalogo}.{esquema}.polizas_test"

try:
    v_challenger = cliente.get_model_version_by_alias(nombre_modelo, "challenger")
except MlflowException:
    dbutils.notebook.exit("No hay challenger: nada que validar")
v_champion = cliente.get_model_version_by_alias(nombre_modelo, "champion")

champion = mlflow.pyfunc.load_model(f"models:/{nombre_modelo}@champion")
challenger = mlflow.pyfunc.load_model(f"models:/{nombre_modelo}@challenger")

version_test = spark.sql(f"DESCRIBE HISTORY {tabla_test} LIMIT 1").collect()[0]["version"]
test = spark.sql(f"SELECT * FROM {tabla_test} VERSION AS OF {version_test}").toPandas()
y = test["ClaimNb"] / test["Exposure"]
w = test["Exposure"]

print("champion v", v_champion.version, "| challenger v", v_challenger.version)

In [0]:
def evaluar(modelo):
    columnas = modelo.metadata.get_input_schema().input_names()
    mu = modelo.predict(test[columnas])
    return mu, {
        "devianza": mean_poisson_deviance(y, mu, sample_weight=w),
        "ae": test["ClaimNb"].sum() / (mu * w).sum(),
    }

mu_ch, m_ch = evaluar(champion)
mu_cl, m_cl = evaluar(challenger)

bandas_edad = [17, 25, 35, 45, 55, 65, 120]
seg = test.assign(
    tramo_edad=pd.cut(test["DrivAge"], bandas_edad).astype(str),
    esp_champion=mu_ch * w,
    esp_challenger=mu_cl * w,
)
segmentos = pd.concat([
    seg.groupby(col)[["Exposure", "ClaimNb", "esp_champion", "esp_challenger"]].sum()
    .reset_index().rename(columns={col: "valor"}).assign(segmento=col)
    for col in ["Area", "tramo_edad"]
])
segmentos["ae_champion"] = segmentos["ClaimNb"] / segmentos["esp_champion"]
segmentos["ae_challenger"] = segmentos["ClaimNb"] / segmentos["esp_challenger"]

print(m_ch, m_cl)
display(segmentos)

In [0]:
mejora = (m_ch["devianza"] - m_cl["devianza"]) / m_ch["devianza"]
calibrado = abs(m_cl["ae"] - 1) <= tolerancia_ae
aprobado = mejora >= mejora_minima and calibrado
decision = "aprobado" if aprobado else "rechazado"

mlflow.set_experiment(experimento_validacion)
with mlflow.start_run(run_name=f"v{v_challenger.version}_vs_v{v_champion.version}"):
    mlflow.log_params({
        "version_champion": v_champion.version,
        "version_challenger": v_challenger.version,
        "tabla_test": tabla_test,
        "version_test": version_test,
        "mejora_minima": mejora_minima,
        "tolerancia_ae": tolerancia_ae,
    })
    mlflow.log_metrics({
        "devianza_champion": m_ch["devianza"],
        "devianza_challenger": m_cl["devianza"],
        "ae_champion": m_ch["ae"],
        "ae_challenger": m_cl["ae"],
        "mejora_relativa": mejora,
    })
    mlflow.set_tag("decision", decision)
    mlflow.log_table(segmentos, "ae_segmentos.json")

cliente.set_model_version_tag(nombre_modelo, v_challenger.version, "validacion", decision)
if aprobado:
    cliente.set_registered_model_alias(nombre_modelo, "anterior", v_champion.version)
    cliente.set_registered_model_alias(nombre_modelo, "champion", v_challenger.version)
cliente.delete_registered_model_alias(nombre_modelo, "challenger")

print(decision, f"| mejora {mejora:.2%} | A/E challenger {m_cl['ae']:.4f}")